# Idea `xau-prior-atr-complete`

This notebook is a **live representation** of Python in `themis.ask`. Run All on the Themis (.venv) kernel. It does not reimplement ATR.

English: *On Gold, for the last month, how many times the ATR is completed from the previous day, give me the notebook, and the html report*

**Ask, not a trade.** Logic: `atr_path_days`, `atr_complete_table`, `atr_complete_rivals`. Families (`strategies/*`, fill, metrics) stay shared and unused here.

Unnamed keys (frozen in `ATR_COMPLETE_RIVALS`): ATR 14 vs 20; complete = daily range vs from-open. Window: last calendar month of loaded bars.


In [ ]:
from pathlib import Path
import os

here = Path.cwd()
repo = here
while repo != repo.parent and not (repo / "docs" / "open-spec.md").exists():
    repo = repo.parent
os.environ["THEMIS_ROOT"] = str(repo)
os.chdir(repo)

from themis.data import load_from_spec
from themis.spec import load_spec
from themis.ask import (
    atr_path_days,
    atr_complete_rivals,
    plot_range_vs_prior_atr,
)

SPEC = repo / 'research/runs/20260829T025701Z-xau-atr-complete-atr14-range-bdc0a206/spec.yaml'
spec = load_spec(SPEC)
spec["id"], spec.get("measure"), spec.get("instrument")


## 1. Series

`themis.data` loads the frozen spec. Binance `XAUUSDT` perp, not COMEX.


In [ ]:
series = load_from_spec(spec, root=repo, network=False)
ohlc = series.df
series.identity


## 2. Path frame (Python)

`atr_path_days` builds UTC daily OHLC, range, from-open, ATR, **prior_atr = atr.shift(1)**.


In [ ]:
path = atr_path_days(ohlc, atr_n=14)
path[["open", "high", "low", "close", "day_range", "from_open", "atr", "prior_atr"]].tail(8)


## 3. Rivals (Python catalog)

`atr_complete_rivals` applies `ATR_COMPLETE_RIVALS` and the last-calendar-month window.


In [ ]:
summary, tables = atr_complete_rivals(ohlc)
summary


## 4. One rival, day by day


In [ ]:
july = tables[(14, "day_range")]
july[["open", "high", "low", "close", "prior_atr", "day_range", "from_open", "complete", "range_over_atr"]]


In [ ]:
plot_range_vs_prior_atr(
    july,
    title="XAUUSDT daily range vs prior-day ATR(14) — last calendar month",
)


Chat quotes **run folders**. This notebook calls the same functions. If they disagree, the folder wins.

Do not multiply `complete_rate` by R. Screen is weak on n≈30.
